# World Cup 2026 — simulação com o XI custom (CSV)

Simula a Copa usando os **XIs titulares atualizados** de `world_cup_2026_starting_xi_custom_players.csv` (stats de jogador por seleção), **juntados com a forma do time (win rate)** do histórico (gold). Os 3 modelos LightGBM já treinados são reutilizados: resultado (W/D/L) + gols do mandante e do visitante (0..5+).


In [2]:
%load_ext autoreload
%autoreload 2


In [3]:
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

from world_cup_features_fifa import (
    get_training_feature_columns,
    build_team_feature_store,
    build_lineup_feature_store,
    prepare_match_row,
)
from world_cup_simulation import predict_match_full, simulate_world_cup


In [4]:
gold_path = "/Users/ormenessevinicius/Library/CloudStorage/OneDrive-TheBostonConsultingGroup,Inc/Documents/GitHub/analise_times/football_analysis/data/gold_fifa_partidas/part-0.parquet"
fixtures_path = "world_cup_2026_games.csv"
xi_csv_path = "world_cup_2026_starting_xi_custom_players.csv"

gold = pd.read_parquet(gold_path)
fixtures = pd.read_csv(fixtures_path)
xi = pd.read_csv(xi_csv_path)
feature_columns = get_training_feature_columns(gold)
print("gold:", gold.shape, "| fixtures:", fixtures.shape, "| XI custom:", xi.shape,
      "| seleções:", xi["nationality"].nunique())


gold: (59970, 329) | fixtures: (104, 12) | XI custom: (528, 14) | seleções: 48


In [12]:
gold

,match_id,match_date,snapshot_date,competition_code,competition_group,match_type,season,home_team,away_team,home_goals,...,away_xi_overall_mean_MID,away_xi_overall_mean_FWD,away_squad_source,away_team_avg_score,away_team_attack_score,away_team_mid_score,away_team_def_score,away_team_gk_score,home_win_rate_last10,away_win_rate_last10
0,INT-Friendlies|2015|2015-01-04|Bahrain|Jordan,2015-01-04,2015-01-02,INT-Friendlies,national_friendly,nation,2015,Bahrain,Jordan,1,...,61.500000,NaN,nationality_top,59.818182,NaN,58.500000,62.000000,NaN,0.0,0.0
1,INT-Friendlies|2015|2015-01-04|Bahrain|Jordan:...,2015-01-04,2015-01-02,INT-Friendlies,national_friendly,nation,2015,Jordan,Bahrain,0,...,NaN,NaN,nationality_top,68.000000,NaN,NaN,74.000000,NaN,0.0,0.0
2,INT-Friendlies|2015|2015-01-04|Iran|Iraq,2015-01-04,2015-01-02,INT-Friendlies,national_friendly,nation,2015,Iran,Iraq,1,...,63.000000,NaN,nationality_top,62.571429,NaN,63.125000,63.750000,62.0,0.0,0.0
3,INT-Friendlies|2015|2015-01-04|Iran|Iraq:mirror,2015-01-04,2015-01-02,INT-Friendlies,national_friendly,nation,2015,Iraq,Iran,0,...,75.000000,63.000000,nationality_top,66.500000,67.333333,71.000000,66.500000,67.0,0.0,0.0
4,INT-Friendlies|2015|2015-01-04|South Africa|Za...,2015-01-04,2015-01-02,INT-Friendlies,national_friendly,nation,2015,South Africa,Zambia,1,...,68.000000,71.000000,nationality_top,69.250000,77.666667,65.750000,74.500000,NaN,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59965,INT-Friendlies|2026|2026-06-10|Portugal|Nigeri...,2026-06-10,2023-01-13,INT-Friendlies,national_friendly,nation,2026,Nigeria,Portugal,1,...,83.666667,86.666667,nation_official,84.636364,83.444444,80.666667,78.500000,81.0,0.3,0.6
59966,INT-World Cup|2026|2026-06-11|Mexico|South Africa,2026-06-11,2023-01-13,INT-World Cup,world_cup,nation,2026,Mexico,South Africa,2,...,72.666667,72.333333,nationality_top,71.818182,72.000000,69.333333,63.333333,72.0,0.6,0.3
59967,INT-World Cup|2026|2026-06-11|Mexico|South Afr...,2026-06-11,2023-01-13,INT-World Cup,world_cup,nation,2026,South Africa,Mexico,0,...,78.000000,79.666667,nation_official,78.000000,79.444444,72.666667,74.875000,80.0,0.3,0.6
59968,INT-World Cup|2026|2026-06-11|South Korea|Czec...,2026-06-11,2023-01-13,INT-World Cup,world_cup,nation,2026,South Korea,Czech Republic,2,...,77.250000,80.333333,nation_official,77.363636,78.555556,73.000000,74.833333,80.0,0.6,0.8


In [5]:
# 3 modelos já treinados (resultado + gols casa/fora)
result_model = lgb.Booster(model_file="fifa_best_model.lgb")
home_goals_model = lgb.Booster(model_file="fifa_home_goals_model.lgb")
away_goals_model = lgb.Booster(model_file="fifa_away_goals_model.lgb")

le = LabelEncoder()
le.classes_ = np.array(json.load(open("fifa_best_model_meta.json"))["classes"])
goal_labels = json.load(open("fifa_home_goals_model_meta.json"))["labels"]
goal_models = (home_goals_model, away_goals_model, goal_labels, goal_labels)
print("classes:", list(le.classes_), "| gols:", goal_labels)


classes: [np.str_('D'), np.str_('L'), np.str_('W')] | gols: ['0', '1', '2', '3', '4', '5+']


## Feature store: XI do CSV + win rate do histórico

`build_lineup_feature_store` monta, por seleção, as features de jogador (slots p01..p11, agregados do XI, scores por setor) a partir do CSV e junta a forma do time (`win_rate_last10`) vinda do histórico (gold).


In [6]:
hist_store = build_team_feature_store(gold, feature_columns)        # win rate / forma (histórico)
store = build_lineup_feature_store(xi, feature_columns, hist_store)  # jogadores do CSV + forma
print("seleções no store custom:", len(store.team_features))
bz = store.team_features["Brazil"]
print("ex. Brazil -> xi_overall_mean:", round(float(bz["xi_overall_mean"]),1),
      "| team_attack_score:", round(float(bz["team_attack_score"]),1),
      "| win_rate_last10:", round(float(bz["win_rate_last10"]),3))


seleções no store custom: 48
ex. Brazil -> xi_overall_mean: 83.9 | team_attack_score: 82.4 | win_rate_last10: 0.4


In [7]:
# Sanidade: uma partida (XI do CSV) -> resultado + distribuição de gols
full = predict_match_full("Brazil", "Germany", result_model, store, le, goal_models=goal_models)
print("Brazil vs Germany:", {k: round(v,3) for k,v in full["proba"].items()},
      "| placar provável", full["goals"]["home_goals"], "-", full["goals"]["away_goals"])
pd.DataFrame({"Brazil (home)": full["goals"]["home_dist"], "Germany (away)": full["goals"]["away_dist"]})


Brazil vs Germany: {np.str_('D'): np.float64(0.335), np.str_('L'): np.float64(0.419), np.str_('W'): np.float64(0.246)} | placar provável 1 - 1


,Brazil (home),Germany (away)
0,0.266583,0.227834
1,0.402925,0.272355
2,0.208431,0.239470
3,0.070840,0.096436
4,0.040292,0.144219
5+,0.010930,0.019686


## Simular a Copa do Mundo (com os XIs do CSV)


In [8]:
simulation = simulate_world_cup(
    fixtures_df=fixtures,
    model=result_model,
    store=store,
    label_encoder=le,
    rng_seed=42,
    goal_models=goal_models,   # também prevê o placar de cada jogo
)
print("🏆 Campeão:", simulation["champion"])


🏆 Campeão: France


In [9]:
simulation["knockout_matches"]


,MatchNumber,RoundNumber,Stage,HomeTeam,AwayTeam,result,winner,most_likely_result,most_likely_winner,proba_W,proba_D,proba_L,home_goals,away_goals
0,73,4,Round of 32,Korea Republic,Canada,W,Korea Republic,W,Korea Republic,0.362851,0.351843,0.285306,1,1
1,74,4,Round of 32,Germany,Paraguay,W,Germany,W,Germany,0.676316,0.193722,0.129962,2,0
2,75,4,Round of 32,Netherlands,Morocco,W,Netherlands,W,Netherlands,0.465416,0.229048,0.305536,1,1
3,76,4,Round of 32,Brazil,Sweden,W,Brazil,W,Brazil,0.488086,0.291295,0.220619,1,0
4,77,4,Round of 32,France,IR Iran,W,France,W,France,0.726375,0.158140,0.115485,2,0
5,78,4,Round of 32,Côte d'Ivoire,Senegal,L,Senegal,L,Senegal,0.248362,0.312535,0.439102,1,1
6,79,4,Round of 32,Czechia,Scotland,W,Czechia,W,Czechia,0.381802,0.310105,0.308093,1,1
7,80,4,Round of 32,England,Cabo Verde,W,England,W,England,0.658726,0.190947,0.150327,2,0
8,81,4,Round of 32,USA,Bosnia and Herzegovina,W,USA,W,USA,0.481871,0.304033,0.214097,1,1
9,82,4,Round of 32,Belgium,South Africa,W,Belgium,W,Belgium,0.625974,0.228343,0.145683,1,0


In [10]:
simulation["standings"]


,team,pts,gf,ga,gd,wins,draws,losses,Group,rank
0,Czechia,9,3,0,3,3,0,0,Group A,1
1,Korea Republic,6,2,1,1,2,0,1,Group A,2
2,South Africa,3,1,2,-1,1,0,2,Group A,3
3,Mexico,0,0,3,-3,0,0,3,Group A,4
4,Switzerland,9,3,0,3,3,0,0,Group B,1
5,Canada,6,2,1,1,2,0,1,Group B,2
6,Bosnia and Herzegovina,3,1,2,-1,1,0,2,Group B,3
7,Qatar,0,0,3,-3,0,0,3,Group B,4
8,Brazil,9,3,0,3,3,0,0,Group C,1
9,Morocco,6,2,1,1,2,0,1,Group C,2


In [11]:
simulation["group_matches"]


,MatchNumber,RoundNumber,Stage,HomeTeam,AwayTeam,result,winner,most_likely_result,most_likely_winner,proba_W,proba_D,proba_L,home_goals,away_goals
0,1,1,Group A,Mexico,South Africa,L,South Africa,L,South Africa,0.326216,0.328397,0.345387,1,1
1,2,1,Group A,Korea Republic,Czechia,L,Czechia,L,Czechia,0.333756,0.325688,0.340556,1,1
2,25,2,Group A,Czechia,South Africa,W,Czechia,W,Czechia,0.376670,0.343372,0.279957,1,1
3,28,2,Group A,Mexico,Korea Republic,L,Korea Republic,L,Korea Republic,0.290323,0.307227,0.402450,1,1
4,53,3,Group A,Czechia,Mexico,W,Czechia,W,Czechia,0.401778,0.314296,0.283926,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,22,1,Group L,England,Croatia,W,England,W,England,0.610484,0.218691,0.170826,2,1
68,45,2,Group L,England,Ghana,W,England,W,England,0.717494,0.165887,0.116619,2,0
69,46,2,Group L,Panama,Croatia,L,Croatia,L,Croatia,0.208117,0.265623,0.526260,0,1
70,67,3,Group L,Panama,England,L,England,L,England,0.100787,0.164737,0.734475,0,2
